# Experimentos para mejorar el modelo
**Problema:** la misma receta dio 0,69 y 0,60 de exactitud en prueba (variación por semilla), y las fotos de 1920×1080 se reducen a 224×224, donde las manchas pequeñas de la roya casi desaparecen.

**Qué hace este notebook**
1. Entrena la receta ganadora (4b) con varios **tamaños de entrada** (224, 320, 448 px) y **3 semillas** cada uno.
2. Compara usando **solo validación** (media ± desviación y un *conjunto* de las 3 semillas). El test no se toca hasta el final.
3. Con el mejor tamaño arma un **conjunto de modelos** (promedia las 3 semillas), lo evalúa **una sola vez** en test y lo exporta a ONNX para la web.

**Es reanudable:** cada corrida terminada se guarda. Si Colab se desconecta o hay un error, vuelva a ejecutar el notebook y salta lo ya entrenado (en Colab, con `USAR_DRIVE = True`, los modelos parciales quedan en su Google Drive).

> Activar GPU en Colab: Runtime → Change runtime type → T4 GPU. **Sin GPU (por ejemplo en un portátil) el entrenamiento completo puede tardar muchas horas**; use `FAST_MODE` o el script `scripts/entrenar_experimentos.py --prueba` para verificar la instalación.
> No es seguro que mejore: el resultado puede ser que 224 px sea suficiente.

In [ ]:
import os, subprocess, sys
from pathlib import Path

EN_COLAB = os.path.exists("/content")
USAR_DRIVE = True        # solo Colab: guarda los modelos parciales en tu Google Drive (sobreviven a una desconexión)

if EN_COLAB:
    REPO = "/content/ia-Kaggle-Coffee"
    if not os.path.isdir(REPO):
        subprocess.run(["git", "clone", "-q", "-b", "main", "https://github.com/JavierAnd0/ia-Kaggle-Coffee", REPO], check=True)
    os.chdir(REPO)

# Subir hasta la carpeta del proyecto (la que tiene src/ y data_splits/), desde donde se abra el notebook.
raiz = Path.cwd()
while not ((raiz / "src").is_dir() and (raiz / "data_splits").is_dir()) and raiz != raiz.parent:
    raiz = raiz.parent
if not (raiz / "src").is_dir():
    raise SystemExit("No encuentro la carpeta del proyecto (con src/ y data_splits/). Abra el notebook dentro del repositorio.")
os.chdir(raiz)
sys.path.insert(0, str(raiz))
print("Proyecto:", raiz)

# Instala solo lo que falte
faltan = []
for modulo, paquete in (("keras_tuner", "keras-tuner"), ("onnx", "onnx"), ("tf2onnx", "tf2onnx"), ("onnxruntime", "onnxruntime"),
                        ("sklearn", "scikit-learn"), ("pandas", "pandas"), ("tensorflow", "tensorflow")):
    try:
        __import__(modulo)
    except ImportError:
        faltan.append(paquete)
if faltan:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltan], check=True)

import json, numpy as np, pandas as pd, tensorflow as tf
from src import config, generators, tuning, experiments, export

DIR_MODELOS = None
if EN_COLAB and USAR_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DIR_MODELOS = "/content/drive/MyDrive/cafe_checkpoints"
        print("Modelos parciales en Google Drive:", DIR_MODELOS)
    except Exception as e:
        print("No se pudo montar Drive; se guardará en el disco temporal de Colab:", e)

FAST_MODE = False                 # True = prueba rápida (pocas épocas, 2 tamaños, 2 semillas); sus resultados no sirven para decidir
TAMANOS = (224, 320, 448) if not FAST_MODE else (224, 320)
SEMILLAS = (0, 1, 2) if not FAST_MODE else (0, 1)
EPOCAS, EPOCAS_FT = (30, 10) if not FAST_MODE else (2, 1)
TAMANO_FINAL = None               # None = elegir el mejor por validación; o fijar uno (p. ej. 320)
tf.keras.utils.set_random_seed(config.SEED)
print("GPU:", tf.config.list_physical_devices("GPU") or "NINGUNA (será muy lento)")

In [ ]:
tr = pd.read_csv(config.SPLITS_DIR / "train.csv")
va = pd.read_csv(config.SPLITS_DIR / "val.csv")
te = pd.read_csv(config.SPLITS_DIR / "test.csv")
li = generators.make_label_index(tr["label"])
hp = tuning.load_best_hp()
print(li, len(tr), len(va), len(te)); hp.values

## 1. Rejilla: tamaño × semilla (solo validación)
Cada línea impresa es un modelo terminado. Si se corta, vuelva a ejecutar esta celda: las corridas ya hechas se reutilizan.

In [ ]:
corridas, resumen = experiments.correr_rejilla(hp, tr, va, li, tamanos=TAMANOS, semillas=SEMILLAS,
                                              epocas=EPOCAS, epocas_ft=EPOCAS_FT, dir_modelos=DIR_MODELOS)
config.RESULTS_DIR.mkdir(exist_ok=True)
corridas.to_csv(config.RESULTS_DIR / "experimentos_corridas.csv", index=False)
resumen.to_csv(config.RESULTS_DIR / "experimentos_resumen.csv")
resumen.round(3)

**Cómo leer la tabla:** `f1_media ± f1_desv` es el F1 macro de las corridas sueltas; `f1_conjunto` es el del promedio de las semillas.
Si la diferencia entre tamaños es menor que la desviación, **no hay evidencia de que un tamaño sea mejor**: en ese caso conviene elegir el más pequeño (más rápido en la web).

In [ ]:
mejor = int(resumen["f1_conjunto"].idxmax()) if TAMANO_FINAL is None else TAMANO_FINAL
print("Tamaño elegido:", mejor)
resumen.loc[[mejor]].round(3)

## 2. Conjunto de modelos del tamaño elegido: test (una sola vez) y exportación para la web
Genera `web/model/` (ONNX + etiquetas + ficha + paridad) y `metricas_web.json`, que la web usa para mostrar cifras **del modelo exportado**.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
r = experiments.finalizar(hp, tr, va, te, li, resumen, SEMILLAS, DIR_MODELOS, config.ROOT / "web" / "model", TAMANO_FINAL)
nombres = sorted(li, key=li.get)
print("Versión:", r["version"], "| diferencia Keras vs ONNX:", r["diferencia_onnx"])
print("VALIDACIÓN:", r["metricas"]["val"])
print("TEST      :", r["metricas"]["test"])
pred = r["p_test"].argmax(1)
print(classification_report(r["y_test"], pred, target_names=nombres, zero_division=0))
print(pd.DataFrame(confusion_matrix(r["y_test"], pred), index=nombres, columns=nombres))

In [ ]:
zip_path = export.empaquetar(config.ROOT / "modelo_cafe.zip", [config.ROOT / "web" / "model"],
                             [config.RESULTS_DIR / "experimentos_corridas.csv", config.RESULTS_DIR / "experimentos_resumen.csv"])
print("Paquete listo:", zip_path)
try:                                   # descarga automática solo en Colab
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    print("Fuera de Colab: el archivo está en esa ruta; envíelo o copie web/model/ al repositorio.")

## 3. Qué hacer con el resultado
- Copie `web/model/*` del zip a `web/model/` del repositorio y súbalo. La web lo carga sola (lee el tamaño de entrada de `model_card.json`) y muestra las cifras de `metricas_web.json`.
- Compare `f1_conjunto` con el modelo actual (validación: exactitud 0,73, F1 macro 0,68). Si **no mejora**, no lo reemplace: quédese con el actual.
- Pendiente (no se resuelve aquí): validar con café arábica (BRACOL o JMuBEN) y con fotos de cultivos colombianos.